# 📖 Notebook 2: Typeahead and Autocomplete

When you type "coff" into Facebook's search bar, it instantly suggests "coffee", "coffee shop", "coffee recipe". This is **typeahead** — predicting what you want before you finish typing.

In this notebook, we'll build autocomplete from scratch and then see how Elasticsearch handles it in production.

## Learning Objectives

By the end of this notebook, you'll understand:
- The difference between **search** and **autocomplete**
- How **prefix matching** works and why naive approaches are slow
- What a **prefix trie** is and how it enables fast completions
- How to use Elasticsearch **edge n-grams** for autocomplete
- How to use the Elasticsearch **completion suggester** for typeahead
- How **fuzzy matching** rescues users who make typos

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 06-system-designs/fb-post-search
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
from elasticsearch import Elasticsearch, helpers
from collections import defaultdict
import time
import re

DB_CONFIG = {
    "host": "localhost",
    "port": 55433,
    "database": "fb_post_search",
    "user": "demo",
    "password": "demo"
}

es = Elasticsearch("http://localhost:9200")

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Verify connections
try:
    conn = get_db()
    conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    es.info()
    print("✅ Elasticsearch connected")
except Exception as e:
    print(f"❌ Elasticsearch: {e}")

## 🤔 Search vs Autocomplete

Search and autocomplete solve different problems:

| | Search | Autocomplete |
|---|--------|-------------|
| **When** | User presses Enter | User is still typing |
| **Input** | Complete words | Partial prefixes |
| **Speed needed** | < 500ms | < 100ms (feels instant) |
| **Returns** | Full documents | Suggested completions |
| **Example** | "coffee" → posts about coffee | "coff" → ["coffee", "coffee shop"] |

Autocomplete needs to be **extremely fast** because it fires on every keystroke. If the user types "coffee" (6 letters), that's potentially 6 separate queries — one for "c", "co", "cof", "coff", "coffe", "coffee".

## 🐌 Approach 1: SQL `LIKE 'prefix%'`

The simplest approach: use SQL `LIKE` with a prefix pattern. Unlike `LIKE '%word%'` (which scans everything), `LIKE 'word%'` (prefix only) **can use a B-tree index**.

But there's a catch — we're matching against post *content*, not individual words. We'd need to search for the prefix at the start of any word in the text.

In [ ]:
# Naive approach: LIKE with prefix pattern

def autocomplete_like(prefix):
    """Find posts containing words starting with the prefix."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    # Match prefix at start of content or after a space
    cur.execute(
        """SELECT id, content, like_count
           FROM posts
           WHERE content ILIKE %s OR content ILIKE %s
           ORDER BY like_count DESC
           LIMIT 5""",
        (f"{prefix}%", f"% {prefix}%")
    )
    results = cur.fetchall()
    conn.close()
    return results

# Simulate typing "coff" one letter at a time
for prefix in ["c", "co", "cof", "coff"]:
    start = time.time()
    results = autocomplete_like(prefix)
    elapsed = (time.time() - start) * 1000
    print(f"  '{prefix}' → {len(results)} results in {elapsed:.1f}ms")

print()
print("⚠️ Problems with this approach:")
print("   1. ILIKE with leading % still causes full table scans")
print("   2. Returns posts, not word suggestions")
print("   3. Too slow for per-keystroke queries at scale")

## 🌳 Approach 2: Prefix Trie (In-Memory)

A **trie** (pronounced "try") is a tree-shaped data structure designed for prefix lookups. Each node represents one character, and paths from root to leaf spell out complete words.

```
            (root)
           /      \
          c        p
         / \        \
        o   a        y
       /     \        \
      f       t       t
     /                 \
    f                   h
   /                     \
  e                       o
  |                       |
  e ✓ (coffee)            n ✓ (python)
```

To find all words starting with "cof", just walk c → o → f and collect all completions below that node.

**Time complexity**: O(prefix_length) to reach the node, then O(results) to collect completions.  
Compare to scanning all words: O(total_words).

In [ ]:
# Build a prefix trie from our posts

class TrieNode:
    """A single node in the trie."""
    def __init__(self):
        self.children = {}        # char → TrieNode
        self.is_word = False      # marks end of a complete word
        self.count = 0            # how many posts contain this word

class Trie:
    """Prefix trie for autocomplete suggestions."""
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word, count=1):
        """Add a word to the trie."""
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_word = True
        node.count += count

    def _find_node(self, prefix):
        """Navigate to the node representing the prefix."""
        node = self.root
        for char in prefix:
            if char not in node.children:
                return None
            node = node.children[char]
        return node

    def autocomplete(self, prefix, limit=5):
        """Find top completions for a prefix, sorted by frequency."""
        node = self._find_node(prefix)
        if not node:
            return []

        # Collect all words below this node using DFS
        results = []
        stack = [(node, prefix)]
        while stack:
            current, word = stack.pop()
            if current.is_word:
                results.append((word, current.count))
            for char, child in current.children.items():
                stack.append((child, word + char))

        # Sort by frequency (most common first) and return top results
        results.sort(key=lambda x: -x[1])
        return results[:limit]

print("✅ Trie class defined. Let's populate it from our posts!")

In [ ]:
# Populate the trie with words from all posts

STOP_WORDS = {
    'the', 'a', 'an', 'is', 'it', 'in', 'on', 'at', 'to', 'for',
    'of', 'and', 'or', 'but', 'not', 'with', 'this', 'that', 'was',
    'are', 'be', 'has', 'had', 'have', 'do', 'does', 'did', 'from',
    'my', 'i', 'me', 'we', 'you', 'he', 'she', 'they', 'so', 'if'
}

conn = get_db()
cur = conn.cursor()
cur.execute("SELECT content FROM posts")
all_content = cur.fetchall()
conn.close()

trie = Trie()
word_counts = defaultdict(int)

# Count word frequency across all posts
for (content,) in all_content:
    words = re.findall(r'[a-z0-9]+', content.lower())
    for word in words:
        if word not in STOP_WORDS and len(word) > 1:
            word_counts[word] += 1

# Insert into trie
for word, count in word_counts.items():
    trie.insert(word, count)

print(f"✅ Built trie with {len(word_counts)} unique words from {len(all_content)} posts")

In [ ]:
# Test autocomplete with the trie

print("🌳 Trie Autocomplete Demo")
print("=" * 50)

prefixes = ["cof", "py", "sea", "tay", "mac"]

for prefix in prefixes:
    start = time.time()
    suggestions = trie.autocomplete(prefix, limit=5)
    elapsed = (time.time() - start) * 1000
    print(f"\n  '{prefix}' → ({elapsed:.3f}ms)")
    for word, count in suggestions:
        print(f"    {word} (appears in {count} posts)")

In [ ]:
# Simulate a user typing "coffee" — measure each keystroke

print("⌨️ Simulating user typing 'coffee':")
print("=" * 50)

for i in range(1, len("coffee") + 1):
    prefix = "coffee"[:i]
    start = time.time()
    suggestions = trie.autocomplete(prefix, limit=3)
    elapsed = (time.time() - start) * 1000
    suggestion_strs = [f"{w}({c})" for w, c in suggestions]
    print(f"  User types '{prefix}' → {suggestion_strs}  ({elapsed:.3f}ms)")

print()
print("💡 Every keystroke returns results in < 1ms!")
print("   This is why tries are perfect for typeahead.")

## 🔍 Approach 3: Elasticsearch Edge N-Grams

In production, we don't want to build and maintain our own trie. Elasticsearch provides **edge n-grams** — a way to index prefixes of words at index time so prefix queries are blazing fast at search time.

How edge n-grams work for the word "coffee":

```
Original word: "coffee"
Edge n-grams:  "c", "co", "cof", "coff", "coffe", "coffee"
```

Each prefix is stored in the inverted index. When the user types "cof", Elasticsearch finds it immediately because "cof" is already a token in the index.

**The trade-off**: We use more storage (more tokens per word), but searches are much faster.

In [ ]:
# Create an Elasticsearch index with edge n-gram analyzer

AUTOCOMPLETE_INDEX = "posts_autocomplete"

if es.indices.exists(index=AUTOCOMPLETE_INDEX):
    es.indices.delete(index=AUTOCOMPLETE_INDEX)

es.indices.create(
    index=AUTOCOMPLETE_INDEX,
    body={
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "analysis": {
                "filter": {
                    "edge_ngram_filter": {
                        "type": "edge_ngram",
                        "min_gram": 1,    # index from 1 character
                        "max_gram": 20    # up to 20 characters
                    }
                },
                "analyzer": {
                    "autocomplete_index": {
                        "type": "custom",
                        "tokenizer": "standard",
                        "filter": ["lowercase", "edge_ngram_filter"]
                    },
                    "autocomplete_search": {
                        "type": "custom",
                        "tokenizer": "standard",
                        "filter": ["lowercase"]
                    }
                }
            }
        },
        "mappings": {
            "properties": {
                "content": {
                    "type": "text",
                    "analyzer": "autocomplete_index",           # index with edge n-grams
                    "search_analyzer": "autocomplete_search"    # search without n-grams
                },
                "user_id":    {"type": "integer"},
                "like_count": {"type": "integer"},
                "created_at": {"type": "date"}
            }
        }
    }
)

print("✅ Created autocomplete index with edge n-gram analyzer")
print()
print("How it works:")
print('  At index time: "coffee" → ["c", "co", "cof", "coff", "coffe", "coffee"]')
print('  At search time: "cof" → just looks up "cof" (exact match in index)')

In [ ]:
# Let's see the tokens Elasticsearch generates

result = es.indices.analyze(
    index=AUTOCOMPLETE_INDEX,
    body={
        "analyzer": "autocomplete_index",
        "text": "coffee"
    }
)

tokens = [t["token"] for t in result["tokens"]]
print(f"Edge n-grams for 'coffee': {tokens}")
print()
print("Each of these becomes a key in the inverted index.")
print("When someone types 'cof', it matches the 'cof' token instantly!")

In [ ]:
# Bulk index posts into the autocomplete index

conn = get_db()
cur = conn.cursor()
cur.execute("SELECT id, user_id, content, like_count, created_at FROM posts")
rows = cur.fetchall()
conn.close()

actions = []
for row in rows:
    actions.append({
        "_index": AUTOCOMPLETE_INDEX,
        "_id": row[0],
        "_source": {
            "user_id": row[1],
            "content": row[2],
            "like_count": row[3],
            "created_at": row[4].isoformat() if row[4] else None
        }
    })

success, _ = helpers.bulk(es, actions)
es.indices.refresh(index=AUTOCOMPLETE_INDEX)

print(f"✅ Indexed {success} posts into autocomplete index")

In [ ]:
# Autocomplete search with edge n-grams

def autocomplete_es(prefix):
    """Autocomplete using Elasticsearch edge n-grams."""
    result = es.search(
        index=AUTOCOMPLETE_INDEX,
        body={
            "query": {
                "match": {
                    "content": {
                        "query": prefix,
                        "operator": "and"
                    }
                }
            },
            "size": 5,
            "_source": ["content", "like_count"]
        }
    )
    return result["hits"]["hits"]

# Simulate typing "coff"
print("🔍 Elasticsearch Autocomplete (Edge N-Grams)")
print("=" * 50)

for prefix in ["c", "co", "cof", "coff", "coffe", "coffee"]:
    start = time.time()
    results = autocomplete_es(prefix)
    elapsed = (time.time() - start) * 1000
    print(f"\n  User types '{prefix}' ({elapsed:.1f}ms) → {len(results)} results")
    for hit in results[:2]:
        content = hit['_source']['content'][:60]
        likes = hit['_source']['like_count']
        print(f"    [{likes} likes] {content}...")

## 🏆 Approach 4: Elasticsearch Completion Suggester

For true typeahead (suggesting **words or phrases**, not full documents), Elasticsearch provides a dedicated **completion suggester**. It's optimized for speed using an in-memory finite state transducer (FST).

The completion suggester is what powers search bars like Google's autocomplete — it suggests query completions, not search results.

Let's create an index of popular search terms and use the completion suggester.

In [ ]:
# Create an index for search term suggestions

SUGGEST_INDEX = "search_suggestions"

if es.indices.exists(index=SUGGEST_INDEX):
    es.indices.delete(index=SUGGEST_INDEX)

es.indices.create(
    index=SUGGEST_INDEX,
    body={
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0
        },
        "mappings": {
            "properties": {
                "suggest": {
                    "type": "completion"    # special type for typeahead
                },
                "popularity": {
                    "type": "integer"
                }
            }
        }
    }
)

print("✅ Created search suggestions index with completion field")

In [ ]:
# Extract popular terms from posts and index them as suggestions

# Count how often each word appears across all posts
# Words that appear more often are better suggestions
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT content FROM posts")
all_content = cur.fetchall()
conn.close()

word_freq = defaultdict(int)
for (content,) in all_content:
    words = re.findall(r'[a-z]+', content.lower())
    for word in words:
        if word not in STOP_WORDS and len(word) > 2:
            word_freq[word] += 1

# Also extract common two-word phrases (bigrams)
bigram_freq = defaultdict(int)
for (content,) in all_content:
    words = re.findall(r'[a-z]+', content.lower())
    words = [w for w in words if w not in STOP_WORDS and len(w) > 2]
    for i in range(len(words) - 1):
        bigram = f"{words[i]} {words[i+1]}"
        bigram_freq[bigram] += 1

# Index top suggestions (single words + popular bigrams)
actions = []

# Top 200 single words
top_words = sorted(word_freq.items(), key=lambda x: -x[1])[:200]
for word, freq in top_words:
    actions.append({
        "_index": SUGGEST_INDEX,
        "_source": {
            "suggest": {
                "input": word,
                "weight": freq     # more popular = ranked higher
            },
            "popularity": freq
        }
    })

# Top 100 bigrams
top_bigrams = sorted(bigram_freq.items(), key=lambda x: -x[1])[:100]
for bigram, freq in top_bigrams:
    if freq >= 2:  # only include bigrams that appear at least twice
        actions.append({
            "_index": SUGGEST_INDEX,
            "_source": {
                "suggest": {
                    "input": bigram,
                    "weight": freq
                },
                "popularity": freq
            }
        })

success, _ = helpers.bulk(es, actions)
es.indices.refresh(index=SUGGEST_INDEX)

print(f"✅ Indexed {success} search suggestions")
print(f"   - {len(top_words)} single-word suggestions")
print(f"   - {sum(1 for _, f in top_bigrams if f >= 2)} bigram suggestions")

In [ ]:
# Use the completion suggester for typeahead

def typeahead(prefix):
    """Get typeahead suggestions using Elasticsearch completion suggester."""
    result = es.search(
        index=SUGGEST_INDEX,
        body={
            "suggest": {
                "post-suggest": {
                    "prefix": prefix,
                    "completion": {
                        "field": "suggest",
                        "size": 5,
                        "skip_duplicates": True
                    }
                }
            }
        }
    )
    suggestions = result["suggest"]["post-suggest"][0]["options"]
    return [(s["text"], s["_score"]) for s in suggestions]

print("🏆 Completion Suggester Typeahead")
print("=" * 50)

test_prefixes = ["cof", "py", "sea", "tay", "mac", "run"]

for prefix in test_prefixes:
    start = time.time()
    suggestions = typeahead(prefix)
    elapsed = (time.time() - start) * 1000
    suggestion_strs = [f"{text}" for text, score in suggestions]
    print(f"  '{prefix}' → {suggestion_strs}  ({elapsed:.1f}ms)")

In [ ]:
# Full typing simulation — compare all approaches

print("⌨️ Full Comparison: User types 'coffee'")
print("=" * 70)
print(f"{'Prefix':<10} {'SQL LIKE':<15} {'Trie':<15} {'ES N-Gram':<15} {'Suggester':<15}")
print("-" * 70)

for i in range(2, len("coffee") + 1):
    prefix = "coffee"[:i]

    # SQL LIKE
    start = time.time()
    autocomplete_like(prefix)
    t_like = (time.time() - start) * 1000

    # Trie
    start = time.time()
    trie.autocomplete(prefix)
    t_trie = (time.time() - start) * 1000

    # ES edge n-gram
    start = time.time()
    autocomplete_es(prefix)
    t_es = (time.time() - start) * 1000

    # Completion suggester
    start = time.time()
    typeahead(prefix)
    t_suggest = (time.time() - start) * 1000

    print(f"  '{prefix}'    {t_like:>8.1f}ms     {t_trie:>8.3f}ms     {t_es:>8.1f}ms     {t_suggest:>8.1f}ms")

print()
print("💡 The trie is fastest (in-memory, no network). But it doesn't scale")
print("   across servers. ES completion suggester is the production choice.")

## 🙃 Bonus: Typo-Tolerant Suggestions (Fuzzy Matching)

Real users make typos. If someone types `cofee` (missing an `f`) our strict prefix lookups return **nothing** — a terrible experience.

The fix is **fuzzy matching**: allow the query to differ from the indexed token by a small number of character edits (insertions, deletions, substitutions). This distance is called **Levenshtein edit distance**.

```
edit_distance("cofee",  "coffee") = 1   # one insertion
edit_distance("cffee",  "coffee") = 1   # one insertion
edit_distance("kaffee", "coffee") = 2   # two substitutions
```

Elasticsearch's completion suggester supports this out of the box via a `fuzzy` option. Google, Facebook, and Amazon all do this — that's why searching for `tayor swfit` still surfaces Taylor Swift.

In [ ]:
# Typo-tolerant typeahead using the completion suggester's fuzzy option

def typeahead_fuzzy(prefix, fuzziness="AUTO"):
    """Typeahead that tolerates small typos in the prefix."""
    result = es.search(
        index=SUGGEST_INDEX,
        body={
            "suggest": {
                "post-suggest": {
                    "prefix": prefix,
                    "completion": {
                        "field": "suggest",
                        "size": 5,
                        "skip_duplicates": True,
                        "fuzzy": {
                            # 'AUTO' = 0 edits for 1–2 chars, 1 edit for 3–5 chars, 2 edits for 6+
                            "fuzziness": fuzziness
                        }
                    }
                }
            }
        }
    )
    return [s["text"] for s in result["suggest"]["post-suggest"][0]["options"]]

# Compare strict vs fuzzy for several realistic typos
print("🙃 Strict vs Fuzzy Typeahead (simulating common typos)")
print("=" * 60)
print(f"{'Typed':<12} {'Strict':<25} {'Fuzzy':<25}")
print("-" * 60)

for typo in ["cofee", "cffee", "pythn", "tayor", "machien", "searh"]:
    strict = [t for t, _ in typeahead(typo)]
    fuzzy  = typeahead_fuzzy(typo)
    print(f"  {typo:<10} {str(strict):<25} {str(fuzzy):<25}")

print()
print("💡 Fuzzy matching = small latency cost, huge UX win.")
print("   Tune fuzziness carefully — too tolerant and irrelevant results leak in.")

## 🏗️ How This Maps to FB Post Search Design

In the actual system design:

1. **Popular search terms** are pre-computed from search analytics and stored in the completion suggester
2. **Edge n-grams** are used for content-based prefix matching as a fallback
3. **Results are cached** aggressively — if 1000 users type "tay", the suggestions are the same for all of them
4. **A CDN** can cache suggestion responses at the edge for < 10ms latency

```
User types "tay"
     │
     ▼
  CDN Cache ──hit──→ Return cached suggestions
     │ miss
     ▼
  Search Service
     │
     ▼
  Completion Suggester (ES)
     │
     ▼
  ["taylor swift", "tacos", "taylor"]
```

## 🧹 Cleanup

In [ ]:
# Keep indices for Notebook 3. Uncomment to delete:

# es.indices.delete(index=AUTOCOMPLETE_INDEX)
# es.indices.delete(index=SUGGEST_INDEX)
# print("🧹 Deleted autocomplete indices")

print("✅ Indices kept for Notebook 3.")

## 📚 Summary

### Key Takeaways

1. **Autocomplete must be < 100ms** — it fires on every keystroke. Speed is everything.
2. **Prefix tries** are the classic data structure — O(prefix_length) lookup time. Great for in-memory, single-server use.
3. **Edge n-grams** pre-compute prefixes at index time so search is a simple lookup. Trade storage for speed.
4. **Completion suggesters** are Elasticsearch's purpose-built typeahead solution. Uses an in-memory FST for maximum speed.
5. **Cache aggressively** — autocomplete queries are highly repetitive. CDN + in-memory caching eliminates most requests.
6. **Tolerate typos** — fuzzy matching (edit distance ≤ 1 or 2) keeps the search bar helpful even when fingers slip.

### When to Use Each Approach

| Approach | Best For |
|----------|----------|
| SQL LIKE | Tiny datasets (< 1K rows) |
| Prefix Trie | In-memory, single server, educational |
| Edge N-Grams | Content-based autocomplete |
| Completion Suggester | Production typeahead (search bars) |

### Next Up

In **Notebook 3**, we'll tackle **search ranking and relevance** — how to sort results by recency, popularity, and a combination of both, just like Facebook does.